<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
# --- Setup (run once at the top) ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"  # or your own repo
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane is Refresh / Content Opportunity Scoring.**

The ML task type is ranking / scoring.The question is “which pages should an editor look at first?”, not “will this one page decline?” (that would be classification) and not “what groups of pages exist?” (clustering). We need a continuous priority score so we can sort every page and hand the editor a ranked queue. Ranking/scoring is the right fit for that decision.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The target is a binary label: is this page declining?**

In the starter data it is defined as:

is_declining_label = 1 when trend_direction == "down", else 0.That label comes from an observed outcome — the change in impressions between the most recent 30 days and the previous 30 days — not from a product rule or hand-written score.Later (with the full warehouse) we can tighten this further by using a true forward window (e.g. “did impressions fall in the next 30 days?”). For now the observed trend direction is a solid, honest proxy.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True).round(3))
print("Base rate of declining pages:", round(df["is_declining_label"].mean(), 3))

is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64
Base rate of declining pages: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Precision@50 (and Precision@100 as a secondary check).
Meaning: of the top 50 pages the model ranks highest for refresh review, what fraction are actually declining?Why this metric: an editor only has time for a short list. Getting the top of the queue right matters more than overall accuracy across all 30k pages.
The starter pipeline already uses Precision@50, so we can compare any new score directly against the hand-written baseline (approx 0.24) and the random-forest result (approx 0.68–0.74).

A “good” result is a clear lift over the baseline — roughly 2–3× on this sample is already meaningful decision support.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


**One row = one content page (one pseudonymized content item).**

The unit of analysis is the page, not the client and not a daily time series.

Below is a small slice of the starter data with the columns that matter for ranking refresh opportunity, plus the target label.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build the target the pipeline uses
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Unit of analysis: one row = one page
cols = [
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update",
    "trend_direction", "is_declining_label"
]

print("Shape (one row = one page):", df.shape)
print()
print("Sample of the unit of analysis:")
display(df[cols].head(8))

print()
print("Label distribution:")
print(df["is_declining_label"].value_counts())
print("Base rate:", round(df["is_declining_label"].mean(), 3))

Shape (one row = one page): (30000, 45)

Sample of the unit of analysis:


,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,0.76,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,0.05,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,0.09,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,0.49,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,0.13,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,8.5,0.03,147,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,7.0,0.00,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,21.2,0.06,445,22,stable,0



Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 0.542


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is easy to write - for example “if trend is down and impressions > 500, put it high on the list.”

That kind of rule already exists in the baseline and only reaches about 24% Precision@50.
The pattern is messier than that:

High-volume pages that are only mildly down can still be more valuable to fix than low-volume pages that are crashing.
Position, CTR, engagement, age, and content type all interact. A page in position 8 with falling impressions is a different opportunity from a page in position 40 with the same percentage drop.


Missingness and content type create systematic gaps that a few if-statements handle poorly.
A learned score can combine those signals without us having to hand-tune every threshold. The starter pipeline already shows a random forest roughly tripling Precision@50 over the hand rule on this sample — that is the concrete reason ML earns its place here. We still keep the rule as the baseline we have to beat.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.